# HPC Parcial 1 — Desarrollo y Experimentación

Este notebook documenta el desarrollo del programa (función de prueba y versiones secuencial/paralela) y la experimentación (medición de tiempos, speedup y eficiencia). El análisis completo de los resultados está en `README.md`.

## 1. Desarrollo del programa

Se implementó la función de prueba:

$$f(x) = \sqrt{x} + x^2 + \sin(x) + \cos(x) + \log(x)$$

en `src/compute.py`, junto con una versión **secuencial** (procesa todo el arreglo en un solo proceso) y la lógica de división en chunks para la versión **paralela** (implementada con `ProcessPoolExecutor` en `src/benchmark.py`).

In [1]:
import sys
sys.path.insert(0, 'src')
from compute import run_sequential
import numpy as np

# Demostracion rapida con un arreglo pequeno
sample = np.linspace(1.0, 10.0, 5)
print('Entrada:', sample)
print('f(x):   ', run_sequential(sample))

Entrada: [ 1.    3.25  5.5   7.75 10.  ]
f(x):    [  3.38177329  12.44160582  34.30308542  65.99246816 104.08177011]


## 2. Experimentación

El benchmark real (multiprocessing con 1, 2 y 4 workers, 3 repeticiones cada uno) se corre como script independiente (`src/benchmark.py`), no dentro de este notebook — `ProcessPoolExecutor` no serializa de forma confiable funciones definidas en el kernel de Jupyter. El resultado se guarda en `results/results.csv` y se carga aquí para mostrarlo.

In [2]:
!python src/benchmark.py --size 20000000 --workers 1 2 4 --repeats 3

workers=1 prueba=1 tiempo=1.2005s


workers=1 prueba=2 tiempo=1.1866s


workers=1 prueba=3 tiempo=1.1992s
-> workers=1 promedio=1.1955s



workers=2 prueba=1 tiempo=1.7956s


workers=2 prueba=2 tiempo=1.7744s


workers=2 prueba=3 tiempo=1.7830s
-> workers=2 promedio=1.7843s



workers=4 prueba=1 tiempo=1.3038s


workers=4 prueba=2 tiempo=1.3385s


workers=4 prueba=3 tiempo=1.3734s
-> workers=4 promedio=1.3386s

Resultados guardados en /home/jonathan-gonzalez/hpc-parcial1/src/../results/results.csv


### Resultados

In [3]:
import csv

with open('results/results.csv', newline='') as f_in:
    reader = csv.DictReader(f_in)
    rows = list(reader)

header = list(rows[0].keys())
print(' | '.join(f'{h:>12}' for h in header))
for row in rows:
    vals = [row['workers']] + [f'{float(row[h]):.4f}' for h in header[1:]]
    print(' | '.join(f'{v:>12}' for v in vals))

     workers |     prueba_1 |     prueba_2 |     prueba_3 |     promedio |      speedup |   eficiencia
           1 |       1.2005 |       1.1866 |       1.1992 |       1.1955 |       1.0000 |       1.0000
           2 |       1.7956 |       1.7744 |       1.7830 |       1.7843 |       0.6700 |       0.3350
           4 |       1.3038 |       1.3385 |       1.3734 |       1.3386 |       0.8931 |       0.2233


## 3. Gráfica de rendimiento

`src/plot_results.py` lee `results/results.csv` (y `results/results_heavy.csv` si existe) y genera, mediante código, tres gráficas por experimento:

1. **Workers vs. tiempo promedio de ejecución** (barras).
2. **Workers vs. speedup**, comparado con el speedup ideal $S_p = p$.
3. **Workers vs. eficiencia**, comparada con la eficiencia ideal $E_p = 1$.

Las imágenes se guardan en `results/performance_plot.png` y `results/performance_plot_heavy.png`.

In [ ]:
!python src/plot_results.py

import os
from IPython.display import Image, display

for png in ['results/performance_plot.png', 'results/performance_plot_heavy.png']:
    if os.path.exists(png):
        display(Image(png))

## 4. Experimento adicional: carga computacional mayor

En el experimento anterior, `f(x)` es barata (NumPy la calcula sobre 20M elementos en ~1.2 s), por lo que el costo de copiar el arreglo a cada proceso y traer los resultados de vuelta domina sobre el ahorro de paralelizar — por eso el speedup fue menor a 1. Para ilustrar el caso opuesto (cómputo dominante), se repite el experimento con `f_heavy`, que hace mucho más trabajo por elemento.

In [4]:
!python src/benchmark.py --heavy --size 2000000 --workers 1 2 4 --repeats 3

workers=1 prueba=1 tiempo=25.2975s


workers=1 prueba=2 tiempo=25.2357s


workers=1 prueba=3 tiempo=25.3283s
-> workers=1 promedio=25.2872s



workers=2 prueba=1 tiempo=17.1378s


workers=2 prueba=2 tiempo=17.1711s


workers=2 prueba=3 tiempo=17.0548s
-> workers=2 promedio=17.1213s



workers=4 prueba=1 tiempo=9.1960s


workers=4 prueba=2 tiempo=9.1794s


workers=4 prueba=3 tiempo=9.1235s
-> workers=4 promedio=9.1663s

Resultados guardados en /home/jonathan-gonzalez/hpc-parcial1/src/../results/results_heavy.csv


### Resultados (carga pesada)

In [5]:
import csv

with open('results/results_heavy.csv', newline='') as f_in:
    reader = csv.DictReader(f_in)
    rows = list(reader)

header = list(rows[0].keys())
print(' | '.join(f'{h:>12}' for h in header))
for row in rows:
    vals = [row['workers']] + [f'{float(row[h]):.4f}' for h in header[1:]]
    print(' | '.join(f'{v:>12}' for v in vals))

     workers |     prueba_1 |     prueba_2 |     prueba_3 |     promedio |      speedup |   eficiencia
           1 |      25.2975 |      25.2357 |      25.3283 |      25.2872 |       1.0000 |       1.0000
           2 |      17.1378 |      17.1711 |      17.0548 |      17.1213 |       1.4769 |       0.7385
           4 |       9.1960 |       9.1794 |       9.1235 |       9.1663 |       2.7587 |       0.6897
